In [1]:
import torch
import matplotlib.pyplot as plt
from torch import nn
print("Gpu enabled: ", torch.cuda.is_available())

In [2]:
# Set random seed for reproducibility
torch.manual_seed(42)

# Generate 500 samples
num_samples = 500

engine_size = torch.empty(num_samples, 1).uniform_(1.0, 5.0)
horsepower = torch.empty(num_samples, 1).uniform_(80.0, 450.0)
age = torch.empty(num_samples, 1).uniform_(0.0, 15.0)
mileage = torch.empty(num_samples, 1).uniform_(5000.0, 150000.0)

# Combine into a single feature matrix X of shape (500, 4)
X_raw = torch.cat([engine_size, horsepower, age, mileage], dim=1)

# True coefficients
# Price = 2500*Engine + 150*HP - 900*Age - 0.12*Mileage + 12000 + Noise
true_weights = torch.tensor([[2500.0], [150.0], [-900.0], [-0.12]])
true_bias = 12000.0
noise = torch.randn(num_samples, 1) * 1500.0

y_raw = torch.matmul(X_raw, true_weights) + true_bias + noise

print("X shape:", X_raw.shape)  # Expected: [500, 4]
print("y shape:", y_raw.shape)  # Expected: [500, 1]
print("\nFirst 3 rows of X_raw:\n", X_raw[:3])
print("\nFirst 3 rows of y_raw:\n", y_raw[:3])

In [3]:
split_percentage = int(0.8* len(X_raw))

x_train, y_train = X_raw[:split_percentage], y_raw[:split_percentage]
x_test, y_test = X_raw[split_percentage:], y_raw[split_percentage:]

len(x_train), len(y_train), len(x_test), len(y_test)

In [4]:
def plot_graph(x_train=x_train, y_train=y_train, x_test=x_test, y_test=y_test, predictions=None, mode="diagonal"):
    plt.figure(figsize=(10, 6))
    
    # Convert PyTorch tensors to CPU/numpy for plotting
    def to_np(tensor):
        return tensor.detach().cpu().numpy() if isinstance(tensor, torch.Tensor) else tensor

    y_test_np = to_np(y_test)
    y_train_np = to_np(y_train)

    if predictions is not None:
        preds_np = to_np(predictions)
        
        if mode == "scatter_index":
            # Plot scattered prices against Sample Index
            train_len = len(y_train_np)
            test_indices = range(train_len, train_len + len(y_test_np))
            
            plt.scatter(range(train_len), y_train_np, c="b", s=20, alpha=0.5, label="Training Prices")
            plt.scatter(test_indices, y_test_np, c="r", s=30, alpha=0.7, label="Actual Test Prices")
            plt.scatter(test_indices, preds_np, c="g", s=30, alpha=0.8, marker="x", label="Model Predicted Prices")
            
            plt.xlabel("Sample Index", fontsize=12)
            plt.ylabel("Car Price ($)", fontsize=12)
            plt.title("Car Price Distribution: Actual vs Predicted across Samples", fontsize=14)
        else:
            # Scatter plot: Actual Price (X-axis) vs Predicted Price (Y-axis)
            plt.scatter(y_test_np, preds_np, c="g", s=25, alpha=0.7, label="Model Predictions")
            
            # 45-degree ideal line where Actual == Predicted
            min_val = min(y_test_np.min(), preds_np.min())
            max_val = max(y_test_np.max(), preds_np.max())
            plt.plot([min_val, max_val], [min_val, max_val], "r--", linewidth=2, label="Ideal Line (Perfect Match)")
            
            plt.xlabel("Actual Car Price ($)", fontsize=12)
            plt.ylabel("Predicted Car Price ($)", fontsize=12)
            plt.title("Actual vs Predicted Car Prices (Diagonal View)", fontsize=14)
    else:
        # Plot distributions of train and test prices
        plt.scatter(range(len(y_train_np)), y_train_np, c="b", s=20, label="Training Prices")
        plt.scatter(range(len(y_train_np), len(y_train_np) + len(y_test_np)), y_test_np, c="r", s=20, label="Test Prices")
        plt.xlabel("Sample Index", fontsize=12)
        plt.ylabel("Car Price ($)", fontsize=12)
        plt.title("Car Price Dataset Distribution", fontsize=14)

    plt.legend(prop={"size": 12})
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.show()

# You can call it in default mode:
plot_graph()


In [5]:
# Device-agnostic code setup
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Feature Normalization (Standardization: zero mean, unit variance)
# Compute mean and std from training set only to prevent data leakage
mean = x_train.mean(dim=0, keepdim=True)
std = x_train.std(dim=0, keepdim=True)

# Standardize train and test features
x_train_norm = (x_train - mean) / std
x_test_norm = (x_test - mean) / std

# Move tensors to target device
x_train_norm = x_train_norm.to(device)
y_train = y_train.to(device)
x_test_norm = x_test_norm.to(device)
y_test = y_test.to(device)

print("x_train_norm shape:", x_train_norm.shape)
print("x_test_norm shape:", x_test_norm.shape)
print("Feature Mean:\n", mean)
print("Feature Std:\n", std)

In [6]:
# Define Multi-feature Linear Regression Model
class CarPricePredictionModel(nn.Module):
    def __init__(self, input_features: int = 4, output_features: int = 1):
        super().__init__()
        # nn.Linear performs y = x * W^T + b
        self.linear_layer = nn.Linear(in_features=input_features, out_features=output_features)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear_layer(x)

# Instantiate the model
torch.manual_seed(42)
model_0 = CarPricePredictionModel(input_features=4, output_features=1).to(device)

print("Model Architecture:\n", model_0)
print("\nInitial Model State Dict:\n", model_0.state_dict())

In [7]:
# Untrained Model Baseline Evaluation
model_0.eval()
with torch.inference_mode():
    untrained_preds = model_0(x_test_norm)

print("First 5 Untrained Predictions:\n", untrained_preds[:5])
print("\nFirst 5 Actual Prices:\n", y_test[:5])

# Plot untrained baseline predictions
plot_graph(predictions=untrained_preds, mode="diagonal")

In [8]:
# Loss Function and Optimizer Setup
loss_fn = nn.L1Loss()  # Mean Absolute Error (MAE)
optimizer = torch.optim.Adam(params=model_0.parameters(), lr=10.0)

print("Loss Function:", loss_fn)
print("Optimizer:", optimizer)

In [9]:
# Set seed for reproducibility
torch.manual_seed(42)

epochs = 3000

# Track metrics
epoch_count = []
train_loss_values = []
test_loss_values = []

# Training & Testing Loop
for epoch in range(1, epochs + 1):
    # --- Training ---
    model_0.train()
    
    # 1. Forward pass
    y_pred = model_0(x_train_norm)
    
    # 2. Calculate training loss
    loss = loss_fn(y_pred, y_train)
    
    # 3. Zero gradients
    optimizer.zero_grad()
    
    # 4. Backpropagation
    loss.backward()
    
    # 5. Optimizer step
    optimizer.step()
    
    # --- Testing ---
    model_0.eval()
    with torch.inference_mode():
        test_pred = model_0(x_test_norm)
        test_loss = loss_fn(test_pred, y_test)
        
    # Log metrics
    if epoch % 100 == 0 or epoch == 1:
        epoch_count.append(epoch)
        train_loss_values.append(loss.item())
        test_loss_values.append(test_loss.item())
        
    if epoch % 500 == 0 or epoch == 1:
        print(f"Epoch: {epoch:4d} | Train Loss (MAE): ${loss.item():.2f} | Test Loss (MAE): ${test_loss.item():.2f}")

In [10]:
# Plot Train and Test Loss curves over epochs
plt.figure(figsize=(10, 6))
plt.plot(epoch_count, train_loss_values, label="Train Loss (MAE)", color="blue", linewidth=2)
plt.plot(epoch_count, test_loss_values, label="Test Loss (MAE)", color="orange", linestyle="--", linewidth=2)
plt.xlabel("Epochs", fontsize=12)
plt.ylabel("Loss ($)", fontsize=12)
plt.title("Training & Test Loss Over Epochs", fontsize=14)
plt.legend(prop={"size": 12})
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

In [11]:
# Final Evaluation on Test Dataset
model_0.eval()
with torch.inference_mode():
    y_preds = model_0(x_test_norm)

final_mae = loss_fn(y_preds, y_test).item()
final_rmse = torch.sqrt(nn.MSELoss()(y_preds, y_test)).item()

print(f"Final Test MAE: ${final_mae:.2f}")
print(f"Final Test RMSE: ${final_rmse:.2f}")

print("\n--- View 1: Actual Price vs. Predicted Price (Diagonal Alignment) ---")
plot_graph(predictions=y_preds, mode="diagonal")

print("\n--- View 2: Sample Index vs. Car Price (Scattered View Comparison) ---")
plot_graph(predictions=y_preds, mode="scatter_index")

In [12]:
# Inspect Trained Model Parameters
print("Learned Model Parameters:")
for name, param in model_0.named_parameters():
    print(f"{name}: {param.data}")

print("\nOriginal True Weights (Engine, HP, Age, Mileage):\n", true_weights.T)
print("Original True Bias:", true_bias)

In [13]:
from pathlib import Path

# Save and Load the Trained Model
MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "car_price_model.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME

print(f"Saving model state dict to: {MODEL_SAVE_PATH}")
torch.save(obj=model_0.state_dict(), f=MODEL_SAVE_PATH)

# Load state_dict into a new model instance
loaded_model = CarPricePredictionModel(input_features=4, output_features=1).to(device)
loaded_model.load_state_dict(torch.load(f=MODEL_SAVE_PATH))

loaded_model.eval()
with torch.inference_mode():
    loaded_preds = loaded_model(x_test_norm)

print("Loaded model predictions match original model:", torch.equal(y_preds, loaded_preds))